# Contents

Builds contructible measures and verifies their usability.

In [1]:
%load_ext sql
%sql duckdb:///../ma_quality.duckdb
%config SqlMagic.displaylimit = 35

Connecting to 'duckdb:///../ma_quality.duckdb'

In [2]:
import datetime as dt
import pandas as pd
from sqlalchemy import create_engine, text

In [3]:
windows = []
for y in [2022, 2023, 2024, 2025]:
    windows.append((dt.date(y,1,1), dt.date(y,12,31), y, "year_end"))
    windows.append((dt.date(y,1,1), dt.date(y,5,15),  y, "may15"))
windows.append((dt.date(2026,1,1), dt.date(2026,5,15), 2026, "may15"))

eng = create_engine("duckdb:///C:/Users/jacob/ma_quality/ma_quality.duckdb")

In [36]:
%%sql 

SELECT DISTINCT concept, note FROM codebook ORDER BY concept

Running query in 'duckdb:///../ma_quality.duckdb'

concept,note
cbp_diastolic,CBP numerator; <90
cbp_systolic,CBP numerator; <140
col_colonoscopy,MY or 9 years prior
col_crc_excl,CRC history exclusion
col_fit,MY only; lives in observations
col_fobt,MY only
cvd,situation; still ASCVD (D18)
cvd,None
diabetes,base
diabetes,complication (D15)


### Colorectal Cancer Screening (COL) - Not usuable, zero closure curve

#### Building and Verifying Measure

In [ ]:
%%sql

CREATE OR REPLACE MACRO col_e(period_start, as_of, my_end) AS TABLE
WITH
denominator AS (
    SELECT p.id AS patient
    FROM patients p
    WHERE date_diff('year', p.birthdate, period_start)  >= 65 
),
excl AS (
    SELECT DISTINCT c.patient
    FROM conditions c
    JOIN codebook cb
      ON cb.concept = 'col_crc_excl'
     AND c.code = cb.code
    WHERE c.start_date <= my_end                          
),
numerator AS (
    SELECT DISTINCT pr.patient
    FROM procedures pr
    JOIN codebook cb
     ON pr.code = cb.code
    WHERE (cb.concept = 'col_colonoscopy'
              AND pr.start_date >= (date_trunc('year', my_end) - INTERVAL '9 years') 
              AND pr.start_date <= as_of)                                             
       OR (cb.concept = 'col_fobt'
              AND pr.start_date BETWEEN period_start AND as_of)
    UNION
    SELECT DISTINCT o.patient
    FROM observations o
    JOIN codebook cb
     ON cb.concept = 'col_fit'
     AND o.code = cb.code
    WHERE o.date BETWEEN period_start AND as_of
)
SELECT
    d.patient,
    (x.patient IS NOT NULL)                            AS excluded,
    (x.patient IS NULL AND n.patient IS NOT NULL)      AS in_numerator
FROM denominator d
LEFT JOIN excl      x ON x.patient = d.patient
LEFT JOIN numerator n ON n.patient = d.patient;

Running query in 'duckdb:///../ma_quality.duckdb'

Count


In [15]:
%%sql

-- How many new distinct patients does each modality add in the back half of 2022?

WITH
colo_by_may AS (
    SELECT DISTINCT patient FROM procedures
    WHERE code='73761001'
      AND start_date >= (date_trunc('year', DATE '2022-12-31') - INTERVAL '9 years')
      AND start_date <= DATE '2022-05-15'
),
colo_by_eoy AS (
    SELECT DISTINCT patient FROM procedures
    WHERE code='73761001'
      AND start_date >= (date_trunc('year', DATE '2022-12-31') - INTERVAL '9 years')
      AND start_date <= DATE '2022-12-31'
),
fobt_by_may AS (
    SELECT DISTINCT patient FROM procedures
    WHERE code='104435004' AND start_date BETWEEN DATE '2022-01-01' AND DATE '2022-05-15'
),
fobt_by_eoy AS (
    SELECT DISTINCT patient FROM procedures
    WHERE code='104435004' AND start_date BETWEEN DATE '2022-01-01' AND DATE '2022-12-31'
),
fit_by_may AS (
    SELECT DISTINCT patient FROM observations
    WHERE code='57905-2' AND date BETWEEN DATE '2022-01-01' AND DATE '2022-05-15'
),
fit_by_eoy AS (
    SELECT DISTINCT patient FROM observations
    WHERE code='57905-2' AND date BETWEEN DATE '2022-01-01' AND DATE '2022-12-31'
)
SELECT
    (SELECT count(*) FROM colo_by_may) AS colo_may,
    (SELECT count(*) FROM colo_by_eoy) AS colo_eoy,
    (SELECT count(*) FROM fobt_by_may) AS fobt_may,
    (SELECT count(*) FROM fobt_by_eoy) AS fobt_eoy,
    (SELECT count(*) FROM fit_by_may)  AS fit_may,
    (SELECT count(*) FROM fit_by_eoy)  AS fit_eoy;

Running query in 'duckdb:///../ma_quality.duckdb'

colo_may,colo_eoy,fobt_may,fobt_eoy,fit_may,fit_eoy
18971,18972,211,555,211,555


In [16]:
%%sql

-- How many new distinct patients does each modality add in the back half of 2025?

WITH
colo_by_may AS (
    SELECT DISTINCT patient FROM procedures
    WHERE code='73761001'
      AND start_date >= (date_trunc('year', DATE '2025-12-31') - INTERVAL '9 years')
      AND start_date <= DATE '2025-05-15'
),
colo_by_eoy AS (
    SELECT DISTINCT patient FROM procedures
    WHERE code='73761001'
      AND start_date >= (date_trunc('year', DATE '2025-12-31') - INTERVAL '9 years')
      AND start_date <= DATE '2025-12-31'
),
fobt_by_may AS (
    SELECT DISTINCT patient FROM procedures
    WHERE code='104435004' AND start_date BETWEEN DATE '2025-01-01' AND DATE '2025-05-15'
),
fobt_by_eoy AS (
    SELECT DISTINCT patient FROM procedures
    WHERE code='104435004' AND start_date BETWEEN DATE '2025-01-01' AND DATE '2025-12-31'
),
fit_by_may AS (
    SELECT DISTINCT patient FROM observations
    WHERE code='57905-2' AND date BETWEEN DATE '2025-01-01' AND DATE '2025-05-15'
),
fit_by_eoy AS (
    SELECT DISTINCT patient FROM observations
    WHERE code='57905-2' AND date BETWEEN DATE '2025-01-01' AND DATE '2025-12-31'
)
SELECT
    (SELECT count(*) FROM colo_by_may) AS colo_may,
    (SELECT count(*) FROM colo_by_eoy) AS colo_eoy,
    (SELECT count(*) FROM fobt_by_may) AS fobt_may,
    (SELECT count(*) FROM fobt_by_eoy) AS fobt_eoy,
    (SELECT count(*) FROM fit_by_may)  AS fit_may,
    (SELECT count(*) FROM fit_by_eoy)  AS fit_eoy;

Running query in 'duckdb:///../ma_quality.duckdb'

colo_may,colo_eoy,fobt_may,fobt_eoy,fit_may,fit_eoy
18699,18702,197,542,197,542


COL-E cannot be used as a forecasting target because its numerator is dominated by a 9-year colonoscopy lookback that is already near-saturated across the eligible population, so a member's screening status is determined by screening history rather than current-year activity. Only 1 person in 2022 and 3 in 2025 newly qualified via colonoscopy in the entire May-to-December window. The stool-test modalities (FOBT/FIT, which are the same event under two codes) do add ~345 people in the back half of each year, but those members are almost all already colonoscopy-covered, so the measure has effectively no closable mid-year gap and mid-year and year-end rates are identical.

### Eye Exam - Diabetes (EED) - Not usuable, EOY average is too low to be relatively realistic

#### Code Searching and Codebook Insertion

In [5]:
%%sql

SELECT CAST(code AS TEXT) AS code, description,
       COUNT(*) AS rows, COUNT(DISTINCT patient) AS patients,
       MIN(start_date) AS earliest, MAX(start_date) AS latest
FROM procedures
WHERE lower(description) LIKE '%retina%'
   OR lower(description) LIKE '%eye exam%'
   OR lower(description) LIKE '%ophthalm%'
   OR lower(description) LIKE '%fundus%'
   OR lower(description) LIKE '%coherence tomography%'
   OR lower(description) LIKE '%photocoagulation%'
   OR lower(description) LIKE '%dilated%'
GROUP BY 1,2 ORDER BY patients DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

code,description,rows,patients,earliest,latest
722161008,Diabetic retinal eye exam (procedure),75935,4296,2012-02-19,2027-02-13
700070005,Optical coherence tomography of retina (procedure),36826,1737,2012-02-19,2027-02-13
413180006,Pan retinal photocoagulation for diabetes (procedure),364,364,2012-03-14,2027-02-03
397539000,Grid retinal photocoagulation (procedure),807,179,2012-02-29,2027-01-31


In [6]:
%%sql
SELECT
  (SELECT COUNT(DISTINCT patient) FROM procedures WHERE code='722161008') AS retinal_screen_ppl,
  (SELECT COUNT(DISTINCT patient) FROM procedures WHERE code='700070005') AS oct_ppl,
  (SELECT COUNT(*) FROM (
     SELECT patient FROM procedures WHERE code='722161008'
     INTERSECT
     SELECT patient FROM procedures WHERE code='700070005')) AS both_ppl;

Running query in 'duckdb:///../ma_quality.duckdb'

retinal_screen_ppl,oct_ppl,both_ppl
4296,1737,1737


Every patient who gets the optical coherence tomography of retina also gets the diabetic retinal eye exam so adding it adds no new patients but is done for sanity.

In [7]:
%%sql

DELETE FROM codebook WHERE concept IN ('eed_exam');
INSERT INTO codebook VALUES
('eed_exam','722161008','SNOMED','raw_procedures','Diabetic retinal screening','retinal exam — screening'),
('eed_exam','700070005','SNOMED','raw_procedures','Optical coherence tomography of retina','retinal exam — monitoring')
;
SELECT * FROM codebook WHERE concept = 'eed_exam';

Running query in 'duckdb:///../ma_quality.duckdb'

concept,code,code_system,source_table,description,note
eed_exam,722161008,SNOMED,raw_procedures,Diabetic retinal screening,retinal exam — screening
eed_exam,700070005,SNOMED,raw_procedures,Optical coherence tomography of retina,retinal exam — monitoring


#### Building and Verifying Measure

In [ ]:
%%sql

CREATE OR REPLACE MACRO eed(period_start, as_of) AS TABLE
WITH
dm_condition AS (
    SELECT DISTINCT c.patient
    FROM conditions c
    JOIN codebook cb
      ON cb.concept = 'diabetes'
     AND c.code = cb.code
    WHERE c.start_date <= as_of
),

dm_medication AS (
    SELECT DISTINCT m.patient
    FROM medications m
    JOIN codebook cb
      ON cb.concept = 'dm_rx'
     AND m.code = cb.code
    WHERE m.start_date <= as_of
      AND m.start_date >= (date_trunc('year', as_of) - INTERVAL '1 year')
),
denominator AS (
    SELECT p.id AS patient
    FROM patients p
    WHERE p.id IN (SELECT patient FROM dm_condition)
       OR p.id IN (SELECT patient FROM dm_medication)
       AND date_diff('year', p.birthdate, period_start) >= 65 
),

numerator AS (
    SELECT DISTINCT pr.patient
    FROM procedures pr
    JOIN codebook cb
      ON cb.concept = 'eed_exam'
     AND pr.code = cb.code
    WHERE pr.start_date BETWEEN period_start AND as_of
)
SELECT
    d.patient,
    FALSE AS excluded,
    (n.patient IS NOT NULL) AS in_numerator
FROM denominator d
LEFT JOIN numerator n ON n.patient = d.patient;

Running query in 'duckdb:///../ma_quality.duckdb'

Count


In [13]:
rows_eed = []
with eng.begin() as con:
    for ps, ao, yr, kind in windows:
        q = text("""
            SELECT COUNT(*) FILTER (WHERE NOT excluded) AS denom,
                   COUNT(*) FILTER (WHERE NOT excluded AND in_numerator) AS numer
            FROM eed(:ps, :ao)
        """)
        denom, numer = con.execute(q, {"ps": ps, "ao": ao}).fetchone()
        rate = round(100.0 * numer / denom, 1) if denom else None
        rows_eed.append(("EED", yr, kind, ao, denom, numer, rate))

eed_ts = pd.DataFrame(rows_eed, columns=["measure","year","snapshot","as_of",
                                     "denominator","numerator","rate_pct"])
eed_ts.to_sql("eed_timeseries", eng, if_exists="replace", index=False)
eed_ts

c:\Users\jacob\ma_quality\.venv\Lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


,measure,year,snapshot,as_of,denominator,numerator,rate_pct
0,EED,2022,year_end,2022-12-31,9541,3985,41.8
1,EED,2022,may15,2022-05-15,9425,1812,19.2
2,EED,2023,year_end,2023-12-31,9785,4066,41.6
3,EED,2023,may15,2023-05-15,9629,1890,19.6
4,EED,2024,year_end,2024-12-31,9997,4115,41.2
5,EED,2024,may15,2024-05-15,9862,1963,19.9
6,EED,2025,year_end,2025-12-31,10187,4176,41.0
7,EED,2025,may15,2025-05-15,10077,2003,19.9
8,EED,2026,may15,2026-05-15,10257,2045,19.9


A 1-Star rating corresponds to a rate below 60%, and a 2-Star rating to a rate of 60–72%. Observed end-of-year self-closure averages 41%, which is implausibly low relative to a extremly poor performance.

### Blood Sugar Controlled - Not usuable, uncontrolled population is essentially nonexistent 

In [ ]:
%%sql

CREATE OR REPLACE MACRO gsd(period_start, as_of) AS TABLE
WITH
denominator AS (
    SELECT DISTINCT p.id AS patient
    FROM patients p
    JOIN conditions c ON c.patient = p.id
    JOIN codebook cb
      ON cb.concept = 'diabetes'
     AND c.code = cb.code
    WHERE c.start_date <= as_of
        AND date_diff('year', p.birthdate, period_start) >= 65 
),
a1c AS (
    SELECT o.patient,
           o.date,
           TRY_CAST(o.value AS DOUBLE) AS a1c_val
    FROM observations o
    JOIN codebook cb
      ON cb.concept = 'gsd_a1c'
     AND o.code = cb.code
    WHERE o.date BETWEEN period_start AND as_of
      AND TRY_CAST(o.value AS DOUBLE) IS NOT NULL
),
ranked AS (
    SELECT patient, a1c_val,
           ROW_NUMBER() OVER (PARTITION BY patient ORDER BY date DESC, a1c_val ASC) AS rn
    FROM a1c
),
most_recent AS (
    SELECT patient, a1c_val FROM ranked WHERE rn = 1
)
SELECT
    d.patient,
    FALSE AS excluded,
    CASE
        WHEN m.patient IS NULL        THEN FALSE      -- no A1c in window -> not controlled
        WHEN m.a1c_val < 8.0          THEN TRUE
        ELSE FALSE
    END AS in_numerator
FROM denominator d
LEFT JOIN most_recent m ON m.patient = d.patient;

Running query in 'duckdb:///../ma_quality.duckdb'

Count


In [ ]:
rows_cbs = []
with eng.begin() as con:
    for ps, ao, yr, kind in windows:
        q = text("""
            SELECT COUNT(*) FILTER (WHERE NOT excluded) AS denom,
                   COUNT(*) FILTER (WHERE NOT excluded AND in_numerator) AS numer
            FROM gsd(:ps, :ao)
        """)
        denom, numer = con.execute(q, {"ps": ps, "ao": ao}).fetchone()
        rate = round(100.0 * numer / denom, 1) if denom else None
        rows_cbs.append(("GSD", yr, kind, ao, denom, numer, rate))

gsd_ts = pd.DataFrame(rows_cbs, columns=["measure","year","snapshot","as_of",
                                     "denominator","numerator","rate_pct"])
gsd_ts.to_sql("gsd_timeseries", eng, if_exists="replace", index=False)
gsd_ts

,measure,year,snapshot,as_of,denominator,numerator,rate_pct
0,GSD,2022,year_end,2022-12-31,9541,7958,83.4
1,GSD,2022,may15,2022-05-15,9425,4874,51.7
2,GSD,2023,year_end,2023-12-31,9784,8165,83.5
3,GSD,2023,may15,2023-05-15,9628,5017,52.1
4,GSD,2024,year_end,2024-12-31,9997,8330,83.3
5,GSD,2024,may15,2024-05-15,9862,5221,52.9
6,GSD,2025,year_end,2025-12-31,10187,8454,83.0
7,GSD,2025,may15,2025-05-15,10077,5327,52.9
8,GSD,2026,may15,2026-05-15,10257,5461,53.2


In [ ]:
%%sql

WITH may_recent AS (
    SELECT patient, a1c_val FROM (
        SELECT o.patient, TRY_CAST(o.value AS DOUBLE) AS a1c_val,
               ROW_NUMBER() OVER (PARTITION BY o.patient ORDER BY o.date DESC, TRY_CAST(o.value AS DOUBLE) ASC) rn
        FROM observations o
        JOIN codebook cb ON cb.concept='gsd_a1c'  
            AND o.code=cb.code
        WHERE o.date BETWEEN DATE '2025-01-01' AND DATE '2025-05-15'
          AND TRY_CAST(o.value AS DOUBLE) IS NOT NULL
    ) WHERE rn=1
),
eoy_recent AS (
    SELECT patient, a1c_val FROM (
        SELECT o.patient, TRY_CAST(o.value AS DOUBLE) AS a1c_val,
               ROW_NUMBER() OVER (PARTITION BY o.patient ORDER BY o.date DESC, TRY_CAST(o.value AS DOUBLE) ASC) rn
        FROM observations o
        JOIN codebook cb ON cb.concept='gsd_a1c'  
            AND o.code=cb.code
        WHERE o.date BETWEEN DATE '2025-01-01' 
            AND DATE '2025-12-31'
            AND TRY_CAST(o.value AS DOUBLE) IS NOT NULL
    ) WHERE rn=1
),
dm AS (
    SELECT DISTINCT pop.id AS patient FROM population pop
    JOIN conditions c ON c.patient=pop.id
    JOIN codebook cb ON cb.concept='diabetes' 
    AND c.code=cb.code
)
SELECT
    COUNT(*) FILTER (WHERE e.a1c_val < 8.0
                       AND (mr.patient IS NULL))                       AS closed_via_new_test,
    COUNT(*) FILTER (WHERE e.a1c_val < 8.0
                       AND mr.a1c_val >= 8.0)                          AS closed_via_improvement
FROM dm d
JOIN eoy_recent e ON e.patient = d.patient
LEFT JOIN may_recent mr ON mr.patient = d.patient
WHERE e.a1c_val < 8.0;

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

closed_via_new_test,closed_via_improvement
3200,1


In [ ]:
%%sql

WITH diabetics AS (
    SELECT DISTINCT c.patient
    FROM conditions c
    JOIN codebook cb
      ON cb.concept = 'diabetes'
     AND c.code = cb.code
),
a1c AS (
    SELECT o.patient,
           TRY_CAST(o.value AS DOUBLE) AS a1c,
           (d.patient IS NOT NULL) AS is_diabetic
    FROM observations o
    LEFT JOIN diabetics d ON d.patient = o.patient
    WHERE o.code = '4548-4'
      AND TRY_CAST(o.value AS DOUBLE) IS NOT NULL
)
SELECT
    CASE WHEN is_diabetic THEN 'diabetic' ELSE 'non-diabetic' END AS cohort,
    COUNT(*)                                         AS a1c_tests,
    COUNT(DISTINCT patient)                          AS patients,
    ROUND(MIN(a1c),1)                                AS min_val,
    ROUND(AVG(a1c),2)                                AS avg_val,
    ROUND(MAX(a1c),1)                                AS max_val,
    ROUND(quantile_cont(a1c, 0.5),1)                 AS median,
    ROUND(quantile_cont(a1c, 0.9),1)                 AS p90,
    COUNT(*) FILTER (WHERE a1c > 9.0)                AS over_9,
    COUNT(*) FILTER (WHERE a1c < 8.0)                AS under_8
FROM a1c
GROUP BY is_diabetic
ORDER BY cohort;

Running query in 'duckdb:///../ma_quality.duckdb'

cohort,a1c_tests,patients,min_val,avg_val,max_val,median,p90,over_9,under_8
diabetic,580471,9165,2.0,3.95,12.0,3.5,6.0,142,579734
non-diabetic,186247,12809,5.0,6.09,6.4,6.1,6.3,0,186247


The most-recent HbA1c of the diabetic's have a median of 3.5% and only 142 of 580,471 tests above the 9% threshold, 99.98% of members collapse into one outcome class giving zero variance for a model to fit. Only one person in 2025 closes via improvement confirming the synthetic A1c field carries no usable glycemic signal.

### Kidney Health Evaluation (KED) - Usuable

In [ ]:
%%sql

CREATE OR REPLACE MACRO ked(period_start, as_of) AS TABLE
WITH
denominator AS (
    SELECT DISTINCT p.id AS patient
    FROM patients p
    JOIN conditions c ON c.patient = p.id
    JOIN codebook cb
      ON cb.concept = 'diabetes'
     AND c.code = cb.code
    WHERE c.start_date <= as_of
         AND date_diff('year', pt.birthdate, period_start) >= 65 
),
has_egfr AS (
    SELECT DISTINCT o.patient
    FROM observations o
    JOIN codebook cb
      ON cb.concept = 'ked_egfr'
     AND o.code = cb.code
    WHERE o.date BETWEEN period_start AND as_of
),
has_uacr AS (
    SELECT DISTINCT o.patient
    FROM observations o
    JOIN codebook cb
      ON cb.concept = 'ked_uacr'
     AND o.code = cb.code
    WHERE o.date BETWEEN period_start AND as_of
),
numerator AS (
    SELECT g.patient
    FROM has_egfr g
    JOIN has_uacr u ON u.patient = g.patient
)
SELECT
    d.patient,
    FALSE AS excluded,         
    (n.patient IS NOT NULL) AS in_numerator
FROM denominator d
LEFT JOIN numerator n ON n.patient = d.patient;

Running query in 'duckdb:///../ma_quality.duckdb'

Count


In [7]:
rows_ked = []
with eng.begin() as con:
    for ps, ao, yr, kind in windows:
        q = text("""
            SELECT COUNT(*) FILTER (WHERE NOT excluded) AS denom,
                   COUNT(*) FILTER (WHERE NOT excluded AND in_numerator) AS numer
            FROM ked(:ps, :ao)
        """)
        denom, numer = con.execute(q, {"ps": ps, "ao": ao}).fetchone()
        rate = round(100.0 * numer / denom, 1) if denom else None
        rows_ked.append(("KED", yr, kind, ao, denom, numer, rate))

ked_ts = pd.DataFrame(rows_ked, columns=["measure","year","snapshot","as_of",
                                     "denominator","numerator","rate_pct"])
ked_ts.to_sql("ked_timeseries", eng, if_exists="replace", index=False)
ked_ts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

c:\Users\jacob\ma_quality\.venv\Lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


,measure,year,snapshot,as_of,denominator,numerator,rate_pct
0,KED,2022,year_end,2022-12-31,9541,5185,54.3
1,KED,2022,may15,2022-05-15,9425,2906,30.8
2,KED,2023,year_end,2023-12-31,9784,5296,54.1
3,KED,2023,may15,2023-05-15,9628,2998,31.1
4,KED,2024,year_end,2024-12-31,9997,5379,53.8
5,KED,2024,may15,2024-05-15,9862,3119,31.6
6,KED,2025,year_end,2025-12-31,10187,5426,53.3
7,KED,2025,may15,2025-05-15,10077,3163,31.4
8,KED,2026,may15,2026-05-15,10257,3230,31.5


In [8]:
crows_ked = []
with eng.begin() as con:
    for ps, ao, yr, kind in windows:
        q = text("""
            SELECT p.county,
                   COUNT(*) FILTER (WHERE NOT m.excluded)                      AS denom,
                   COUNT(*) FILTER (WHERE NOT m.excluded AND m.in_numerator)   AS numer
            FROM ked(:ps, :ao) m
            JOIN patients p ON p.id = m.patient
            GROUP BY p.county
        """)
        df = pd.read_sql(q, con, params={"ps": ps, "ao": ao})
        df.insert(0, "measure", "KED")
        df["year"] = yr
        df["snapshot"] = kind
        df["as_of"] = ao
        df["rate_pct"] = (100.0 * df["numer"] / df["denom"]).round(1)
        crows_ked.append(df)

ked_county = pd.concat(crows_ked, ignore_index=True)
ked_county.to_sql("ked_county_timeseries", eng, if_exists="replace", index=False)
ked_county

,measure,county,denom,numer,year,snapshot,as_of,rate_pct
0,KED,Polk County,7409,3976,2022,year_end,2022-12-31,53.7
1,KED,Jasper County,418,237,2022,year_end,2022-12-31,56.7
2,KED,Warren County,555,317,2022,year_end,2022-12-31,57.1
3,KED,Dallas County,1159,655,2022,year_end,2022-12-31,56.5
4,KED,Dallas County,1142,368,2022,may15,2022-05-15,32.2
5,KED,Warren County,549,181,2022,may15,2022-05-15,33.0
6,KED,Jasper County,408,126,2022,may15,2022-05-15,30.9
7,KED,Polk County,7326,2231,2022,may15,2022-05-15,30.5
8,KED,Warren County,570,320,2023,year_end,2023-12-31,56.1
9,KED,Jasper County,433,243,2023,year_end,2023-12-31,56.1


### Controlling High Blood Pressure (CBP) - Usuable

In [4]:
%%sql

CREATE OR REPLACE MACRO cbp(period_start, as_of) AS TABLE
WITH
denominator AS (
    SELECT DISTINCT p.id AS patient
    FROM patients p
    JOIN conditions c ON c.patient = p.id
    JOIN codebook cb
      ON cb.concept = 'hypertension'
     AND c.code = cb.code
    WHERE c.start_date <= as_of
        AND date_diff('year', p.birthdate, period_start) BETWEEN 65 AND 85
),
daily AS (
    SELECT
        o.patient,
        o.date::DATE AS reading_date,
        MIN(CASE WHEN cb.concept = 'cbp_systolic'  THEN TRY_CAST(o.value AS DOUBLE) END) AS sys,
        MIN(CASE WHEN cb.concept = 'cbp_diastolic' THEN TRY_CAST(o.value AS DOUBLE) END) AS dia
    FROM observations o
    JOIN codebook cb
      ON cb.concept IN ('cbp_systolic', 'cbp_diastolic')
     AND o.code = cb.code
    JOIN encounters e ON e.id = o.encounter
    WHERE e.encounterclass NOT IN ('inpatient', 'emergency')
      AND o.date BETWEEN period_start AND as_of
    GROUP BY o.patient, o.date::DATE
),
ranked AS (
    SELECT patient, sys, dia,
           ROW_NUMBER() OVER (PARTITION BY patient ORDER BY reading_date DESC) AS rn
    FROM daily
    WHERE sys IS NOT NULL AND dia IS NOT NULL
),
most_recent AS (
    SELECT patient, sys, dia FROM ranked WHERE rn = 1
)
SELECT
    d.patient,
    FALSE AS excluded,
    CASE
        WHEN m.patient IS NULL           THEN FALSE
        WHEN m.sys < 140 AND m.dia < 90  THEN TRUE
        ELSE FALSE
    END AS in_numerator
FROM denominator d
LEFT JOIN most_recent m ON m.patient = d.patient;

Running query in 'duckdb:///../ma_quality.duckdb'

Count


In [5]:
rows_cbp = []
with eng.begin() as con:
    for ps, ao, yr, kind in windows:
        q = text("""
            SELECT COUNT(*) FILTER (WHERE NOT excluded) AS denom,
                   COUNT(*) FILTER (WHERE NOT excluded AND in_numerator) AS numer
            FROM cbp(:ps, :ao)
        """)
        denom, numer = con.execute(q, {"ps": ps, "ao": ao}).fetchone()
        rate = round(100.0 * numer / denom, 1) if denom else None
        rows_cbp.append(("CBP", yr, kind, ao, denom, numer, rate))

cbp_ts = pd.DataFrame(rows_cbp, columns=["measure","year","snapshot","as_of",
                                     "denominator","numerator","rate_pct"])
cbp_ts.to_sql("cbp_timeseries", eng, if_exists="replace", index=False)
cbp_ts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

c:\Users\jacob\ma_quality\.venv\Lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


,measure,year,snapshot,as_of,denominator,numerator,rate_pct
0,CBP,2022,year_end,2022-12-31,11562,9016,78.0
1,CBP,2022,may15,2022-05-15,11510,4820,41.9
2,CBP,2023,year_end,2023-12-31,12282,9681,78.8
3,CBP,2023,may15,2023-05-15,12236,5171,42.3
4,CBP,2024,year_end,2024-12-31,13047,10324,79.1
5,CBP,2024,may15,2024-05-15,12983,5633,43.4
6,CBP,2025,year_end,2025-12-31,13775,10900,79.1
7,CBP,2025,may15,2025-05-15,13720,5979,43.6
8,CBP,2026,may15,2026-05-15,13807,6104,44.2


In [6]:
%%sql

WITH bp_pairs AS (
    SELECT s.patient, s.date,
           TRY_CAST(s.value AS DOUBLE) AS sys,
           TRY_CAST(d.value AS DOUBLE) AS dia
    FROM observations s
    JOIN codebook cbs ON cbs.concept='cbp_systolic'  AND s.code=cbs.code
    JOIN observations d
      ON d.patient=s.patient AND d.encounter=s.encounter AND d.date=s.date
    JOIN codebook cbd ON cbd.concept='cbp_diastolic' AND d.code=cbd.code
),
may_recent AS (
    SELECT patient, sys, dia FROM (
        SELECT patient, sys, dia,
               ROW_NUMBER() OVER (PARTITION BY patient ORDER BY date DESC) rn
        FROM bp_pairs
        WHERE date BETWEEN DATE '2025-01-01' AND DATE '2025-05-15'
    ) WHERE rn=1
),
eoy_recent AS (
    SELECT patient, sys, dia FROM (
        SELECT patient, sys, dia,
               ROW_NUMBER() OVER (PARTITION BY patient ORDER BY date DESC) rn
        FROM bp_pairs
        WHERE date BETWEEN DATE '2025-01-01' AND DATE '2025-12-31'
    ) WHERE rn=1
),
htn AS (
    SELECT DISTINCT pop.id AS patient
    FROM population pop
    JOIN conditions c ON c.patient=pop.id
    JOIN codebook cb ON cb.concept='hypertension' AND c.code=cb.code
    WHERE c.start_date <= DATE '2025-12-31'
)
SELECT
    COUNT(*) FILTER (WHERE mr.patient IS NULL)                          AS closed_via_new_test,
    COUNT(*) FILTER (WHERE NOT (mr.sys < 140 AND mr.dia < 90))          AS closed_via_improvement,
    COUNT(*)                                                            AS total_closers
FROM htn h
JOIN eoy_recent e ON e.patient = h.patient
LEFT JOIN may_recent mr ON mr.patient = h.patient
WHERE (e.sys < 140 AND e.dia < 90)          -- controlled at year-end
  AND (mr.patient IS NULL OR NOT (mr.sys < 140 AND mr.dia < 90));  

Running query in 'duckdb:///../ma_quality.duckdb'

closed_via_new_test,closed_via_improvement,total_closers
4868,171,5039


In [17]:
%%sql

WITH bp_pairs AS (
    SELECT s.patient, s.date,
           TRY_CAST(s.value AS DOUBLE) AS sys,
           TRY_CAST(d.value AS DOUBLE) AS dia
    FROM observations s
    JOIN codebook cbs ON cbs.concept='cbp_systolic'  AND s.code=cbs.code
    JOIN observations d
      ON d.patient=s.patient AND d.encounter=s.encounter AND d.date=s.date
    JOIN codebook cbd ON cbd.concept='cbp_diastolic' AND d.code=cbd.code
),
may_recent AS (
    SELECT patient, sys, dia FROM (
        SELECT patient, sys, dia,
               ROW_NUMBER() OVER (PARTITION BY patient ORDER BY date DESC) rn
        FROM bp_pairs WHERE date BETWEEN DATE '2025-01-01' AND DATE '2025-05-15'
    ) WHERE rn=1
),
eoy_recent AS (
    SELECT patient, sys, dia FROM (
        SELECT patient, sys, dia,
               ROW_NUMBER() OVER (PARTITION BY patient ORDER BY date DESC) rn
        FROM bp_pairs WHERE date BETWEEN DATE '2025-01-01' AND DATE '2025-12-31'
    ) WHERE rn=1
),
htn AS (
    SELECT DISTINCT pop.id AS patient
    FROM population pop
    JOIN conditions c ON c.patient=pop.id
    JOIN codebook cb ON cb.concept='hypertension' AND c.code=cb.code
    WHERE c.start_date <= DATE '2025-12-31'
)
SELECT
    p.county,
    COUNT(*) FILTER (WHERE mr.patient IS NULL)                    AS closed_via_new_test,
    COUNT(*) FILTER (WHERE NOT (mr.sys < 140 AND mr.dia < 90))    AS closed_via_improvement,
    COUNT(*)                                                      AS total_closers
FROM htn h
JOIN eoy_recent e ON e.patient = h.patient
JOIN patients   p ON p.id       = h.patient
LEFT JOIN may_recent mr ON mr.patient = h.patient
WHERE (e.sys < 140 AND e.dia < 90)
  AND (mr.patient IS NULL OR NOT (mr.sys < 140 AND mr.dia < 90))
GROUP BY p.county
ORDER BY closed_via_improvement DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

county,closed_via_new_test,closed_via_improvement,total_closers
Polk County,3801,137,3938
Dallas County,555,22,577
Warren County,288,7,295
Jasper County,224,5,229


### Satin Therapy for Patients with Cardiovacular Disease (SPC) - Unusuable, Rates are pinned above 95% which is too high to be plausible

#### Code Searching and Notebook Insertion

In [18]:
%%sql
-- What codes does the codebook's cvd concept contain vs SPC's 7-code coronary list?
SELECT 'cvd_in_codebook' AS src, CAST(code AS TEXT) AS code, description
FROM codebook WHERE concept='cvd'
ORDER BY code;

Running query in 'duckdb:///../ma_quality.duckdb'

src,code,description
cvd_in_codebook,22298006,Myocardial infarction
cvd_in_codebook,230690007,Cerebrovascular accident
cvd_in_codebook,399211009,History of myocardial infarction
cvd_in_codebook,399261000,History of coronary artery bypass grafting
cvd_in_codebook,401303003,Acute ST elevation myocardial infarction
cvd_in_codebook,401314000,Acute non-ST elevation myocardial infarction
cvd_in_codebook,414545008,Ischemic heart disease
cvd_in_codebook,4557003,Preinfarction syndrome


In [19]:
%%sql
-- What codes does statin_rx contain, and are they intensity-tagged?
SELECT CAST(code AS TEXT) AS code, description, note
FROM codebook WHERE concept='statin_rx'
ORDER BY code;

Running query in 'duckdb:///../ma_quality.duckdb'

code,description,note
197903,Lovastatin 10 MG Oral Tablet,None
197904,Lovastatin 20 MG Oral Tablet,None
197905,Lovastatin 40 MG Oral Tablet,None
198211,Simvastatin 40 MG Oral Tablet,None
200345,Simvastatin 80 MG Oral Tablet,None
259255,Atorvastatin 80 MG Oral Tablet,None
312961,Simvastatin 20 MG Oral Tablet,None
312962,Simvastatin 5 MG Oral Tablet,None
314231,Simvastatin 10 MG Oral Tablet,None
476349,Ezetimibe 10 MG / Simvastatin 20 MG Oral Tablet,combo; contains statin


In [21]:
%%sql

DELETE FROM codebook WHERE concept='spc_statin';
INSERT INTO codebook (concept, code, code_system, source_table, description, note)
SELECT 'spc_statin', code, code_system, source_table, description, 'high/moderate intensity'
FROM codebook
WHERE concept='statin_rx'
  AND code NOT IN ('312962','314231','197903','197904','904458','904467','859424');

Running query in 'duckdb:///../ma_quality.duckdb'

Count


#### Building and Verifying Measure

In [ ]:
%%sql

CREATE OR REPLACE MACRO spc(period_start, as_of) AS TABLE
WITH
eligible AS (
    SELECT pt.id AS patient, pt.gender,
           date_diff('year', pt.birthdate, as_of)
             - CASE WHEN (date_part('month', pt.birthdate), date_part('day', pt.birthdate))
                       > (date_part('month', as_of), date_part('day', as_of))
                    THEN 1 ELSE 0 END AS age_asof
    FROM patients pt
    JOIN population pop ON pop.id = pt.id
),
ascvd AS (
    SELECT DISTINCT c.patient
    FROM conditions c
    JOIN codebook cb ON cb.concept='cvd' AND c.code=cb.code
    WHERE c.start_date <= as_of
),
denominator AS (
    SELECT e.patient
    FROM eligible e
    JOIN ascvd a ON a.patient = e.patient
    WHERE (e.gender='M' AND e.age_asof BETWEEN 21 AND 75)
       OR (e.gender='F' AND e.age_asof BETWEEN 40 AND 75)
),
statin AS (
    SELECT DISTINCT m.patient
    FROM medications m
    JOIN codebook cb ON cb.concept='spc_statin' AND m.code=cb.code
    WHERE m.start_date <= as_of
      AND (m.stop_date IS NULL OR m.stop_date >= period_start)
)
SELECT
    d.patient,
    FALSE AS excluded,        
    (s.patient IS NOT NULL) AS in_numerator
FROM denominator d
LEFT JOIN statin s ON s.patient = d.patient;

Running query in 'duckdb:///../ma_quality.duckdb'

Count


In [ ]:
rows_spc = []
with eng.begin() as con:
    for ps, ao, yr, kind in windows:
        q = text("""
            SELECT COUNT(*) FILTER (WHERE NOT excluded) AS denom,
                   COUNT(*) FILTER (WHERE NOT excluded AND in_numerator) AS numer
            FROM spc(:ps, :ao)
        """)
        denom, numer = con.execute(q, {"ps": ps, "ao": ao}).fetchone()
        rate = round(100.0 * numer / denom, 1) if denom else None
        rows_spc.append(("SPC", yr, kind, ao, denom, numer, rate))

spc_ts = pd.DataFrame(rows_spc, columns=["measure","year","snapshot","as_of",
                                     "denominator","numerator","rate_pct"])
spc_ts.to_sql("spc_timeseries", eng, if_exists="replace", index=False)
spc_ts

c:\Users\jacob\ma_quality\.venv\Lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


,measure,year,snapshot,as_of,denominator,numerator,rate_pct
0,SPC,2022,year_end,2022-12-31,10575,10109,95.6
1,SPC,2022,may15,2022-05-15,10792,10291,95.4
2,SPC,2023,year_end,2023-12-31,10118,9701,95.9
3,SPC,2023,may15,2023-05-15,10469,10014,95.7
4,SPC,2024,year_end,2024-12-31,9618,9246,96.1
5,SPC,2024,may15,2024-05-15,9926,9523,95.9
6,SPC,2025,year_end,2025-12-31,8982,8632,96.1
7,SPC,2025,may15,2025-05-15,9369,9001,96.1
8,SPC,2026,may15,2026-05-15,8756,8407,96.0


Rates are pinned above 95% which is far too high to be plausible

### Taking Diabetes Medication - Unusuable, EOY average is 35%+ lower than the 1 star cut off.

#### Code Searching and Cobebook Insertion

In [24]:
%%sql

SELECT code, description, note
FROM codebook
WHERE concept='dm_rx'
ORDER BY note, code;

Running query in 'duckdb:///../ma_quality.duckdb'

code,description,note
897122,liraglutide 6 MG/ML Pen Injector,GLP-1
860975,24 HR Metformin hydrochloride 500 MG ER Oral Tablet,biguanide
106892,insulin isophane / regular human 70/30 Injectable [Humulin],insulin
311034,insulin regular human 100 UNT/ML Injectable,insulin
865098,Insulin Lispro 100 UNT/ML Injectable [Humalog],insulin


In [ ]:
%%sql
SELECT CAST(code AS TEXT) AS code, description,
       COUNT(*) AS rows, COUNT(DISTINCT patient) AS patients
FROM medications
WHERE lower(description) LIKE '%metformin%'          
   OR lower(description) LIKE '%glipizide%'           
   OR lower(description) LIKE '%glyburide%'
   OR lower(description) LIKE '%glimepiride%'
   OR lower(description) LIKE '%gliclazide%'
   OR lower(description) LIKE '%gl... %'
   OR lower(description) LIKE '%canagliflozin%'      
   OR lower(description) LIKE '%dapagliflozin%'
   OR lower(description) LIKE '%empagliflozin%'
   OR lower(description) LIKE '%ertugliflozin%'
   OR lower(description) LIKE '%sitagliptin%'         
   OR lower(description) LIKE '%saxagliptin%'
   OR lower(description) LIKE '%linagliptin%'
   OR lower(description) LIKE '%alogliptin%'
   OR lower(description) LIKE '%liraglutide%'      
   OR lower(description) LIKE '%semaglutide%'
   OR lower(description) LIKE '%dulaglutide%'
   OR lower(description) LIKE '%exenatide%'
   OR lower(description) LIKE '%pioglitazone%'      
   OR lower(description) LIKE '%rosiglitazone%'
   OR lower(description) LIKE '%acarbose%'           
   OR lower(description) LIKE '%miglitol%'
   OR lower(description) LIKE '%repaglinide%'    
   OR lower(description) LIKE '%nateglinide%'
   OR lower(description) LIKE '%glucophage%'         
   OR lower(description) LIKE '%januvia%'
   OR lower(description) LIKE '%jardiance%'
   OR lower(description) LIKE '%invokana%'
   OR lower(description) LIKE '%farxiga%'
   OR lower(description) LIKE '%ozempic%'
   OR lower(description) LIKE '%trulicity%'
GROUP BY 1,2
ORDER BY patients DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

code,description,rows,patients
860975,24 hr metformin hydrochloride 500 mg extended release oral tablet,196066,2882
897122,3 ml liraglutide 6 mg/ml pen injector,843,40
1373463,canagliflozin 100 mg oral tablet,6,3


In [ ]:
%%sql

DELETE FROM codebook WHERE concept='dm_rx' AND code='1373463';
INSERT INTO codebook (concept, code, code_system, source_table, description, note)
VALUES ('dm_rx','1373463','RxNorm','raw_medications','Canagliflozin 100 MG Oral Tablet','SGLT2');

#### Building and Verifying Macro

In [27]:
%%sql

CREATE OR REPLACE MACRO mac_diabetes(period_start, as_of) AS TABLE
WITH
diab_rx AS (
    SELECT m.patient, m.start_date,
           LEAST(GREATEST(date_diff('day', m.start_date,
                          COALESCE(m.stop_date, m.start_date + INTERVAL 30 DAY)), 0), 90) AS supply_days,
           m.start_date AS fill_date
    FROM medications m
    JOIN codebook cb ON cb.concept='dm_rx' AND m.code=cb.code AND cb.note <> 'insulin'
    WHERE m.start_date BETWEEN period_start AND as_of
),
ipsd AS (
    SELECT patient, MIN(start_date) AS ipsd_date FROM diab_rx GROUP BY patient
),
tx_period AS (
    SELECT i.patient, i.ipsd_date, date_diff('day', i.ipsd_date, as_of) + 1 AS tx_days
    FROM ipsd i
),
two_fills AS (
    SELECT patient FROM (
        SELECT patient, COUNT(DISTINCT fill_date) AS fills FROM diab_rx GROUP BY patient
    ) WHERE fills >= 2
),
insulin_users AS (
    SELECT DISTINCT m.patient
    FROM medications m
    JOIN codebook cb ON cb.concept='dm_rx' AND m.code=cb.code AND cb.note='insulin'
    WHERE m.start_date BETWEEN period_start AND as_of
),
denominator AS (
    SELECT t.patient, t.ipsd_date, t.tx_days
    FROM tx_period t
    JOIN two_fills f ON f.patient = t.patient
    WHERE t.tx_days >= 91
      AND t.patient NOT IN (SELECT patient FROM insulin_users)
),
fill_intervals AS (
    SELECT d.patient,
           GREATEST(r.start_date, d.ipsd_date) AS cov_start,
           LEAST(r.start_date + CAST(r.supply_days AS INTEGER) - 1, as_of) AS cov_end
    FROM denominator d
    JOIN diab_rx r ON r.patient = d.patient
),
covered_days AS (
    SELECT patient, COUNT(*) AS covered
    FROM (
        SELECT DISTINCT fi.patient, fi.cov_start + CAST(gs.i AS INTEGER) AS covered_date
        FROM fill_intervals fi
        CROSS JOIN generate_series(0, GREATEST(date_diff('day', fi.cov_start, fi.cov_end), 0)) AS gs(i)
        WHERE fi.cov_end >= fi.cov_start
    )
    GROUP BY patient
)
SELECT
    d.patient,
    FALSE AS excluded,
    d.tx_days,
    COALESCE(c.covered, 0) AS covered_days,
    ROUND(COALESCE(c.covered,0)::DOUBLE / d.tx_days, 3) AS pdc,
    (COALESCE(c.covered,0)::DOUBLE / d.tx_days >= 0.80) AS in_numerator
FROM denominator d
LEFT JOIN covered_days c ON c.patient = d.patient;

Running query in 'duckdb:///../ma_quality.duckdb'

Count


In [ ]:
rows_md = []
with eng.begin() as con:
    for ps, ao, yr, kind in windows:
        q = text("""
            SELECT COUNT(*) FILTER (WHERE NOT excluded) AS denom,
                   COUNT(*) FILTER (WHERE NOT excluded AND in_numerator) AS numer
            FROM mac_diabetes(:ps, :ao)
        """)
        denom, numer = con.execute(q, {"ps": ps, "ao": ao}).fetchone()
        rate = round(100.0 * numer / denom, 1) if denom else None
        rows_md.append(("MAC_DM", yr, kind, ao, denom, numer, rate))

macdm_ts = pd.DataFrame(rows_md, columns=["measure","year","snapshot","as_of",
                                       "denominator","numerator","rate_pct"])
macdm_ts.to_sql("mac_diabetes_timeseries", eng, if_exists="replace", index=False)
macdm_ts

c:\Users\jacob\ma_quality\.venv\Lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


,measure,year,snapshot,as_of,denominator,numerator,rate_pct
0,MAC_DM,2022,year_end,2022-12-31,444,228,51.4
1,MAC_DM,2022,may15,2022-05-15,185,179,96.8
2,MAC_DM,2023,year_end,2023-12-31,483,245,50.7
3,MAC_DM,2023,may15,2023-05-15,200,190,95.0
4,MAC_DM,2024,year_end,2024-12-31,554,268,48.4
5,MAC_DM,2024,may15,2024-05-15,238,231,97.1
6,MAC_DM,2025,year_end,2025-12-31,546,277,50.7
7,MAC_DM,2025,may15,2025-05-15,224,215,96.0
8,MAC_DM,2026,may15,2026-05-15,245,235,95.9


Rate is far too low to be plausible. EOY average is 35%+ lower than the 1 star cut off.

### Taking Blood Pressure Medication - Unusuable, EOY average is 32%+ lower than the 1 star cut off.

In [34]:
%%sql

SELECT
    CASE
      WHEN lower(description) LIKE '%sacubitril%' THEN 'ARNI (EXCLUDE)'
      WHEN lower(description) LIKE '%pril%'       THEN 'ACE'
      WHEN lower(description) LIKE '%sartan%'     THEN 'ARB'
      WHEN lower(description) LIKE '%aliskiren%'  THEN 'renin inhibitor'
      ELSE 'other'
    END AS ras_class,
    COUNT(DISTINCT code)    AS codes,
    COUNT(DISTINCT patient) AS patients
FROM medications
WHERE lower(description) LIKE '%pril%'
   OR lower(description) LIKE '%sartan%'
   OR lower(description) LIKE '%aliskiren%'
   OR lower(description) LIKE '%sacubitril%'
GROUP BY ras_class
ORDER BY patients DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

ras_class,codes,patients
ACE,29,11351
ARB,23,1898
ARNI (EXCLUDE),2,86


In [ ]:
%%sql

CREATE OR REPLACE MACRO mac_ras(period_start, as_of) AS TABLE
WITH
ras_rx AS (
    SELECT m.patient, m.start_date,
           LEAST(GREATEST(date_diff('day', m.start_date,
                          COALESCE(m.stop_date, m.start_date + INTERVAL 30 DAY)), 0), 90) AS supply_days,
           m.start_date AS fill_date
    FROM medications m
    WHERE m.start_date BETWEEN period_start AND as_of
      AND lower(m.description) NOT LIKE '%sacubitril%'
      AND ( lower(m.description) LIKE '%pril%'
         OR lower(m.description) LIKE '%sartan%'
         OR lower(m.description) LIKE '%aliskiren%' )
),
ipsd AS (  
    SELECT patient, MIN(start_date) AS ipsd_date FROM ras_rx GROUP BY patient
),
tx_period AS (
    SELECT i.patient, i.ipsd_date, date_diff('day', i.ipsd_date, as_of) + 1 AS tx_days
    FROM ipsd i
),
two_fills AS (  
    SELECT patient FROM (
        SELECT patient, COUNT(DISTINCT fill_date) AS fills FROM ras_rx GROUP BY patient
    ) WHERE fills >= 2
),
entresto_users AS (
    SELECT DISTINCT patient FROM medications
    WHERE lower(description) LIKE '%sacubitril%'
      AND start_date BETWEEN period_start AND as_of
),
denominator AS (
    SELECT t.patient, t.ipsd_date, t.tx_days
    FROM tx_period t
    JOIN two_fills f ON f.patient = t.patient
    WHERE t.tx_days >= 91
      AND t.patient NOT IN (SELECT patient FROM entresto_users)
),
fill_intervals AS (
    SELECT d.patient,
           GREATEST(r.start_date, d.ipsd_date) AS cov_start,
           LEAST(r.start_date + CAST(r.supply_days AS INTEGER) - 1, as_of) AS cov_end
    FROM denominator d
    JOIN ras_rx r ON r.patient = d.patient
),
covered_days AS (
    SELECT patient, COUNT(*) AS covered
    FROM (
        SELECT DISTINCT fi.patient, fi.cov_start + CAST(gs.i AS INTEGER) AS covered_date
        FROM fill_intervals fi
        CROSS JOIN generate_series(0, GREATEST(date_diff('day', fi.cov_start, fi.cov_end), 0)) AS gs(i)
        WHERE fi.cov_end >= fi.cov_start
    )
    GROUP BY patient
)
SELECT
    d.patient,
    FALSE AS excluded,
    d.tx_days,
    COALESCE(c.covered, 0) AS covered_days,
    ROUND(COALESCE(c.covered,0)::DOUBLE / d.tx_days, 3) AS pdc,
    (COALESCE(c.covered,0)::DOUBLE / d.tx_days >= 0.80) AS in_numerator
FROM denominator d
LEFT JOIN covered_days c ON c.patient = d.patient;

Running query in 'duckdb:///../ma_quality.duckdb'

Count


In [ ]:
rows_tb = []
with eng.begin() as con:
    for ps, ao, yr, kind in windows:
        q = text("""
            SELECT COUNT(*) FILTER (WHERE NOT excluded) AS denom,
                   COUNT(*) FILTER (WHERE NOT excluded AND in_numerator) AS numer
            FROM mac_ras(:ps, :ao)
        """)
        denom, numer = con.execute(q, {"ps": ps, "ao": ao}).fetchone()
        rate = round(100.0 * numer / denom, 1) if denom else None
        rows_tb.append(("MAC_RAS", yr, kind, ao, denom, numer, rate))

macras_ts = pd.DataFrame(rows_tb, columns=["measure","year","snapshot","as_of",
                                        "denominator","numerator","rate_pct"])
macras_ts.to_sql("mac_ras_timeseries", eng, if_exists="replace", index=False)
macras_ts

c:\Users\jacob\ma_quality\.venv\Lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


,measure,year,snapshot,as_of,denominator,numerator,rate_pct
0,MAC_RAS,2022,year_end,2022-12-31,3410,1939,56.9
1,MAC_RAS,2022,may15,2022-05-15,1489,1449,97.3
2,MAC_RAS,2023,year_end,2023-12-31,3539,2014,56.9
3,MAC_RAS,2023,may15,2023-05-15,1563,1525,97.6
4,MAC_RAS,2024,year_end,2024-12-31,3702,2239,60.5
5,MAC_RAS,2024,may15,2024-05-15,1666,1628,97.7
6,MAC_RAS,2025,year_end,2025-12-31,3817,2240,58.7
7,MAC_RAS,2025,may15,2025-05-15,1733,1697,97.9
8,MAC_RAS,2026,may15,2026-05-15,1856,1795,96.7


### Taking Cholesterol Medication - Unusuable, rates are pinned above 99% which is extremely unplausible

In [37]:
%%sql

SELECT
    CASE
      WHEN lower(description) LIKE '%atorvastatin%'  THEN 'atorvastatin'
      WHEN lower(description) LIKE '%simvastatin%'   THEN 'simvastatin'
      WHEN lower(description) LIKE '%rosuvastatin%'  THEN 'rosuvastatin'
      WHEN lower(description) LIKE '%pravastatin%'   THEN 'pravastatin'
      WHEN lower(description) LIKE '%lovastatin%'    THEN 'lovastatin'
      WHEN lower(description) LIKE '%pitavastatin%'  THEN 'pitavastatin'
      WHEN lower(description) LIKE '%fluvastatin%'   THEN 'fluvastatin'
      ELSE 'other-statin'
    END AS statin_drug,
    COUNT(DISTINCT code)    AS codes,
    COUNT(DISTINCT patient) AS patients
FROM medications
WHERE lower(description) LIKE '%statin%'
GROUP BY statin_drug
ORDER BY patients DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

statin_drug,codes,patients
simvastatin,7,21817
atorvastatin,6,2447
rosuvastatin,6,524
pravastatin,5,486
lovastatin,3,240
pitavastatin,2,21


In [38]:
%%sql

CREATE OR REPLACE MACRO mac_statin(period_start, as_of) AS TABLE
WITH
statin_rx AS (
    SELECT m.patient, m.start_date, m.dispenses,
           m.start_date AS cov_start_raw,
           m.start_date + CAST(COALESCE(m.dispenses,1) * 30 AS INTEGER) - 1 AS cov_end_raw
    FROM medications m
    JOIN codebook cb ON cb.concept='statin_rx' AND m.code=cb.code
    WHERE m.start_date <= as_of
      AND (m.stop_date IS NULL OR m.stop_date >= period_start)
),
disp_totals AS (
    SELECT patient, SUM(COALESCE(dispenses,1)) AS total_dispenses,
           MIN(start_date) AS ipsd_date
    FROM statin_rx GROUP BY patient
),
tx_period AS (
    SELECT patient, ipsd_date,
           date_diff('day', GREATEST(ipsd_date, period_start), as_of) + 1 AS tx_days
    FROM disp_totals
),
denominator AS (   -- >=2 total dispenses, treatment period >=91 days
    SELECT t.patient, t.ipsd_date, t.tx_days
    FROM tx_period t
    JOIN disp_totals d ON d.patient = t.patient
    WHERE d.total_dispenses >= 2
      AND t.tx_days >= 91
),
fill_intervals AS (
    SELECT r.patient,
           GREATEST(r.cov_start_raw, period_start) AS cov_start,
           LEAST(r.cov_end_raw, as_of)             AS cov_end
    FROM statin_rx r
    JOIN denominator d ON d.patient = r.patient
),
covered_days AS (
    SELECT patient, COUNT(*) AS covered
    FROM (
        SELECT DISTINCT fi.patient, fi.cov_start + CAST(gs.i AS INTEGER) AS covered_date
        FROM fill_intervals fi
        CROSS JOIN generate_series(0, GREATEST(date_diff('day', fi.cov_start, fi.cov_end), 0)) AS gs(i)
        WHERE fi.cov_end >= fi.cov_start
    )
    GROUP BY patient
)
SELECT
    d.patient,
    FALSE AS excluded,
    d.tx_days,
    LEAST(COALESCE(c.covered,0), d.tx_days) AS covered_days,
    ROUND(LEAST(COALESCE(c.covered,0), d.tx_days)::DOUBLE / d.tx_days, 3) AS pdc,
    (LEAST(COALESCE(c.covered,0), d.tx_days)::DOUBLE / d.tx_days >= 0.80) AS in_numerator
FROM denominator d
LEFT JOIN covered_days c ON c.patient = d.patient;

Running query in 'duckdb:///../ma_quality.duckdb'

Count


In [ ]:
rows_sat = []
with eng.begin() as con:
    for ps, ao, yr, kind in windows:
        q = text("""
            SELECT COUNT(*) FILTER (WHERE NOT excluded) AS denom,
                   COUNT(*) FILTER (WHERE NOT excluded AND in_numerator) AS numer
            FROM mac_statin(:ps, :ao)
        """)
        denom, numer = con.execute(q, {"ps": ps, "ao": ao}).fetchone()
        rate = round(100.0 * numer / denom, 1) if denom else None
        rows_sat.append(("MAC_STATIN", yr, kind, ao, denom, numer, rate))

macstatin_ts = pd.DataFrame(rows_sat, columns=["measure","year","snapshot","as_of",
                                           "denominator","numerator","rate_pct"])
macstatin_ts.to_sql("mac_statin_timeseries", eng, if_exists="replace", index=False)
macstatin_ts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

c:\Users\jacob\ma_quality\.venv\Lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


,measure,year,snapshot,as_of,denominator,numerator,rate_pct
0,MAC_STATIN,2022,year_end,2022-12-31,19001,18912,99.5
1,MAC_STATIN,2022,may15,2022-05-15,18561,18455,99.4
2,MAC_STATIN,2023,year_end,2023-12-31,19679,19599,99.6
3,MAC_STATIN,2023,may15,2023-05-15,19260,19168,99.5
4,MAC_STATIN,2024,year_end,2024-12-31,20305,20234,99.7
5,MAC_STATIN,2024,may15,2024-05-15,19913,19824,99.6
6,MAC_STATIN,2025,year_end,2025-12-31,20852,20784,99.7
7,MAC_STATIN,2025,may15,2025-05-15,20518,20411,99.5
8,MAC_STATIN,2026,may15,2026-05-15,21052,20968,99.6
